# 07e — Semantic Alignment + TPI

## Objective

This notebook integrates the semantic alignment developed in **07d** directly with the
**Text Preserved Information (TPI)** measure from **07a/07b**.

The complete pipeline is:

```text
Original + Anonymized
        ↓
Contextual term embeddings
        ↓
Cosine similarity matrix
        ↓
Maximum-weight matching
        ↓
Similarity threshold
        ↓
Affected original terms M
        ↓
Information content IC(t)
        ↓
TPI
```

The main output is a content-preservation score for an original/anonymized pair.

This notebook remains an experimental evaluation framework. It does not assume that
the semantic alignment or TPI formulation is optimal.

## Relation to the previous notebooks

### 07a / 07b — TPI

For an original transcript $W_o$:

$$
IC(t)=-\log P(t)
$$

and

$$
TPI
=
1-
\frac{
\sum_{t\in M} IC(t)
}{
\sum_{t\in W_o} IC(t)
}.
$$

The key difficulty is obtaining the affected set $M$.

### 07c — Surface alignment

07c used `SequenceMatcher` to obtain $M$, but lexical matching can incorrectly
flag paraphrases and changes in word order.

### 07d — Semantic alignment

07d replaced this surface alignment with contextual BERT embeddings and
maximum-weight one-to-one matching.

This notebook connects the two stages:

$$
\boxed{
\text{Semantic alignment}
\;\rightarrow\;
M
\;\rightarrow\;
\text{TPI}
}
$$

The threshold remains an explicit experimental parameter.

In [1]:

import importlib.util
from pathlib import Path

import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM


# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

ROOT = Path.cwd().parent
CONTENT_DIR = ROOT / "src" / "content"


def load_module(module_name, file_path):
    """Load a Python module directly from its file path."""
    spec = importlib.util.spec_from_file_location(
        module_name,
        file_path,
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# ------------------------------------------------------------------
# Semantic alignment module
# ------------------------------------------------------------------

semantic_module = load_module(
    "content_semantic_alignment",
    CONTENT_DIR / "semantic_alignment.py",
)

extract_terms = semantic_module.extract_terms
normalize_term = semantic_module.normalize_term
show_terms = semantic_module.show_terms
term_embeddings = semantic_module.term_embeddings
cosine_similarity_matrix = semantic_module.cosine_similarity_matrix
build_similarity_matrix = semantic_module.build_similarity_matrix
semantic_align = semantic_module.semantic_align


# ------------------------------------------------------------------
# TPI module
# ------------------------------------------------------------------

tpi_module = load_module(
    "content_tpi",
    CONTENT_DIR / "tpi.py",
)

estimate_all_term_ic = tpi_module.estimate_all_term_ic
information_content = tpi_module.information_content
masked_term_probabilities = tpi_module.masked_term_probabilities
term_indices = tpi_module.term_indices
total_information_content = tpi_module.total_information_content
tpi_from_spans = tpi_module.tpi_from_spans
whitespace_terms = tpi_module.whitespace_terms
term_subtokens = tpi_module.term_subtokens
build_masked_text = tpi_module.build_masked_text
information_loss_from_spans = tpi_module.information_loss_from_spans


# ------------------------------------------------------------------
# Evaluation module
# ------------------------------------------------------------------

evaluation_module = load_module(
    "content_evaluation",
    CONTENT_DIR / "evaluation.py",
)

tpi_from_alignment = evaluation_module.tpi_from_alignment
evaluate_content_pair = evaluation_module.evaluate_content_pair
evaluate_dataset = evaluation_module.evaluate_dataset


# ------------------------------------------------------------------
# Model and experiment configuration
# ------------------------------------------------------------------

MODEL_NAME = "bert-base-uncased"

# Experimental value established in the 07d controlled tests.
SIMILARITY_THRESHOLD = 0.80

TPI_STRATEGY = "N-TA"
N = 6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# One MLM model provides contextual hidden states and masked-token probabilities.
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
).to(DEVICE)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Semantic similarity threshold:", SIMILARITY_THRESHOLD)
print("TPI strategy:", TPI_STRATEGY)
print("N-TA N:", N)


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 5843.74it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: bert-base-uncased
Device: cpu
Semantic similarity threshold: 0.8
TPI strategy: N-TA
N-TA N: 6


## 1. Term extraction

We keep the same lexical term definition used in 07a/07c/07d.

Punctuation is ignored, while each term keeps its character span.

In [2]:
show_terms("I moved to Geneva in 2022.")

,term,start,end
0,I,0,1
1,moved,2,7
2,to,8,10
3,Geneva,11,17
4,in,18,20
5,2022,21,25


## 2. Contextual embeddings for terms

As in 07d, BERT tokenizes the complete sentence. For each lexical term we
identify the WordPiece tokens overlapping its character span and mean-pool
their contextual hidden states:

$$
e_t=\frac{1}{K}\sum_{k=1}^{K}h_k.
$$

These embeddings are used for alignment, not directly as the information-loss
measure.

## 3. Semantic alignment

For every original/anonymized term pair:

$$
S_{ij}
=
\frac{e_i^{orig}\cdot e_j^{anon}}
{\|e_i^{orig}\|\|e_j^{anon}\|}.
$$

We solve a one-to-one maximum-weight matching problem:

$$
\max_A \sum_{(i,j)\in A}S_{ij}.
$$

Only matched pairs with $S_{ij}\geq	\tau$ are accepted.

Any original term that is not confidently matched is included in \(M\).

In [3]:
def alignment_dataframe(result):
    rows = []

    for item in result["alignments"]:
        rows.append({
            "original_idx": item["original_idx"],
            "original_term": item["original_term"],
            "anonymized_idx": item["anonymized_idx"],
            "anonymized_term": item["anonymized_term"],
            "similarity": round(item["similarity"], 4),
            "status": item["status"],
        })

    for i in result["affected_original_indices"]:
        rows.append({
            "original_idx": i,
            "original_term": result["original_terms"][i]["term"],
            "anonymized_idx": None,
            "anonymized_term": None,
            "similarity": None,
            "status": "affected",
        })

    return (
        pd.DataFrame(rows)
        .sort_values("original_idx")
        .reset_index(drop=True)
    )

## 4. Information content for the original transcript

TPI is computed only from the **original** transcript.

For a term $t$:

$$
IC(t)=-\log P(t\mid context).
$$

As in 07a, a multi-subtoken term uses the minimum probability across its
WordPiece subtokens.

The $N$-TA strategy masks terms in alternating groups.

## 5. TPI from the semantic-alignment output

Once semantic alignment identifies $M$, TPI is computed as:

$$
TPI
=
1-
\frac{
\sum_{t\in M}IC(t)
}{
\sum_{t\in W_o}IC(t)
}.
$$

We also report

$$
L_{TPI}=1-TPI
$$

for compatibility with a future loss formulation.

## 6. Complete semantic-alignment + TPI evaluation

This is the main interface of the notebook.

$$
\boxed{
\text{Original + Anonymized}
\;\rightarrow\;
\text{Semantic alignment}
\;\rightarrow\;
M
\;\rightarrow\;
TPI
}
$$

## 7. Controlled end-to-end experiments

These cases are inherited from 07d. The difference is that we now propagate
the detected affected terms all the way through TPI.

The purpose is to check the complete implementation before applying it to
real data.

In [4]:
cases = {
    "unchanged": (
        "I live in Geneva.",
        "I live in Geneva."
    ),
    "synonym_paraphrase": (
        "I live in Geneva.",
        "I reside in Geneva."
    ),
    "replacement": (
        "I live in Lausanne.",
        "I live in Geneva."
    ),
    "deletion": (
        "I live in Geneva since 2022.",
        "I live in Geneva."
    ),
    "insertion": (
        "I live in Geneva.",
        "I currently live in Geneva."
    ),
    "reordering": (
        "I live in Geneva.",
        "In Geneva I live."
    ),
    "generalization": (
        "I have two brothers.",
        "I have siblings."
    ),
    "multiple_changes": (
        "I moved to Geneva in 2022.",
        "I moved to Switzerland."
    ),
}

controlled_rows = []

for name, (original, anonymized) in cases.items():
    result = evaluate_content_pair(
        original,
        anonymized,
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        similarity_threshold=SIMILARITY_THRESHOLD,
        tpi_strategy=TPI_STRATEGY,
        N=N,
    )

    controlled_rows.append({
        "case": name,
        "affected_terms": ", ".join(
            result["alignment"]["affected_original_terms"]
        ),
        "information_lost": round(
            result["tpi"]["information_lost"], 4
        ),
        "TPI": round(result["TPI"], 4),
        "L_TPI": round(result["L_TPI"], 4),
    })

controlled_results = pd.DataFrame(controlled_rows)
display(controlled_results)

,case,affected_terms,information_lost,TPI,L_TPI
0,unchanged,,0.0000,1.0000,0.0000
1,synonym_paraphrase,,0.0000,1.0000,0.0000
2,replacement,Lausanne,8.9839,0.4075,0.5925
3,deletion,"since, 2022",16.3266,0.4059,0.5941
4,insertion,,0.0000,1.0000,0.0000
5,reordering,"I, live, in, Geneva",12.5777,0.0000,1.0000
6,generalization,"two, brothers",3.4443,0.5126,0.4874
7,multiple_changes,"Geneva, in, 2022",19.5526,0.3249,0.6751


## 8. Inspect one complete example

The output below exposes the complete chain:

$$
\text{alignment}
\;\rightarrow\;
M
\;\rightarrow\;
IC(M)
\;\rightarrow\;
TPI.
$$

This is useful for debugging surprising scores on real anonymized data.

In [5]:
example_original = "I live in Lausanne."
example_anonymized = "I live in Geneva."

example_result = evaluate_content_pair(
    example_original,
    example_anonymized,
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    similarity_threshold=SIMILARITY_THRESHOLD,
    tpi_strategy=TPI_STRATEGY,
    N=N,
)

print("ORIGINAL:")
print(example_original)

print("\nANONYMIZED:")
print(example_anonymized)

print("\nALIGNMENT:")
display(alignment_dataframe(example_result["alignment"]))

print("\nAFFECTED TERMS:")
print(example_result["alignment"]["affected_original_terms"])

print("\nAFFECTED INFORMATION:")
display(example_result["tpi"]["affected_terms"])

print("\nTIC(original):", example_result["tpi"]["TIC_original"])
print("Information lost:", example_result["tpi"]["information_lost"])
print("TPI:", example_result["TPI"])
print("L_TPI:", example_result["L_TPI"])

ORIGINAL:
I live in Lausanne.

ANONYMIZED:
I live in Geneva.

ALIGNMENT:


,original_idx,original_term,anonymized_idx,anonymized_term,similarity,status
0,0,I,0.0,I,0.9799,exact_match
1,1,live,1.0,live,0.9767,exact_match
2,2,in,2.0,in,0.9589,exact_match
3,3,Lausanne,NaN,NaN,NaN,affected



AFFECTED TERMS:
['Lausanne']

AFFECTED INFORMATION:


,index,term,probability,IC
3,3,Lausanne,0.000125,8.983905



TIC(original): 15.16280385915622
Information lost: 8.983904966107774
TPI: 0.40750371438177324
L_TPI: 0.5924962856182268


## 9. Sanity checks

The implementation should preserve the properties already checked in 07b.

In particular:

$$
M=\varnothing \Rightarrow TPI=1
$$

and adding affected terms cannot increase TPI.

In [6]:
unchanged_result = evaluate_content_pair(
    "I live in Geneva.",
    "I live in Geneva.",
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    similarity_threshold=SIMILARITY_THRESHOLD,
    tpi_strategy=TPI_STRATEGY,
    N=N,
)

assert unchanged_result["alignment"]["affected_original_indices"] == []
assert np.isclose(unchanged_result["TPI"], 1.0)

print("✓ Unchanged example gives TPI = 1.")

✓ Unchanged example gives TPI = 1.


In [7]:
test_text = "I moved to Geneva in 2022."

ic_test = estimate_all_term_ic(
    test_text,
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    strategy=TPI_STRATEGY,
    N=N,
)

def tpi_for_indices(ic_table, indices):
    tic = float(ic_table["IC"].sum())
    affected_ic = float(
        ic_table[
            ic_table["index"].isin(set(indices))
        ]["IC"].sum()
    )
    return 1.0 - affected_ic / tic

tpi_empty = tpi_for_indices(ic_test, [])
tpi_one = tpi_for_indices(ic_test, [3])
tpi_two = tpi_for_indices(ic_test, [3, 4])

assert tpi_empty >= tpi_one >= tpi_two

print("TPI(M = ∅):", round(tpi_empty, 4))
print("TPI(M = {3}):", round(tpi_one, 4))
print("TPI(M = {3,4}):", round(tpi_two, 4))
print("✓ Monotonicity check passed.")

TPI(M = ∅): 1.0
TPI(M = {3}): 0.7538
TPI(M = {3,4}): 0.7506
✓ Monotonicity check passed.


## 10. Applying the pipeline to real data

The next stage is to evaluate paired original/anonymized transcripts.

Expected input:

| original | anonymized |
|---|---|
| original transcript 1 | anonymized transcript 1 |
| original transcript 2 | anonymized transcript 2 |
| ... | ... |

The function below keeps the intermediate affected terms as well as the TPI
score so that surprising examples can be inspected manually.

## 11. Interpretation and next step

At this stage, TPI is treated as an **information-preservation component**,
not as a complete semantic content metric.

The content evaluation therefore remains decomposed into:

$$
\boxed{
\text{Semantic Similarity}
+
\text{NLI}
+
\text{TPI}
}
$$

We do **not** fix arbitrary weights for a final combined loss yet.

The immediate next step is empirical evaluation on real original/anonymized
data and analysis of where the three components agree or disagree.

In particular, we want to determine whether TPI adds information that is not
captured by semantic similarity and NLI.